# RL Memory Filter Agent - Interactive Development Notebook

This notebook provides an interactive environment for developing, testing, and visualizing the **RL Memory Filter Agent** for the ARMPA (Adaptive Retrieval with Memory-augmented Planning Agent) system.

## Overview

The RL Filter Agent learns to **intelligently select which memories to include** in the agent's context, balancing:
- **Relevance**: Memories should help solve the current task
- **Efficiency**: Don't overload context with too many memories
- **Adaptability**: Adjust memory count based on agent uncertainty (entropy)

## Notebook Structure

1. **Setup & Data Loading** - Load trajectories, configure environment
2. **Memory System Analysis** - Understand how memories are stored and retrieved
3. **RL Formulation** - Visualize state/action/reward structure
4. **Neural Network Testing** - Test MemoryFilterPolicy with sample inputs
5. **Entropy Analysis** - Understand adaptive memory selection
6. **Training Data Format** - Examine collected RL training data
7. **Training Loop** - Set up PPO training
8. **Evaluation** - Compare filter vs baselines

## 1. Setup & Environment Configuration

In [9]:
# Core imports
import os
import sys
import pickle
import json
from pathlib import Path
from typing import List, Dict, Tuple
from dotenv import load_dotenv

# Data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch for RL
import torch
import torch.nn as nn

# Load environment variables
load_dotenv(override=True)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Auto-reload modules for development
%load_ext autoreload
%autoreload 2
%cd webarena

# Add project paths
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'webarena'))

print("✓ Environment configured")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[Errno 2] No such file or directory: 'webarena'
/Users/manoloalvarez/playground/ARMPA/webarena
✓ Environment configured
  Python: 3.10.19
  PyTorch: 2.9.0
  CUDA available: False
  MPS available: True


/Users/manoloalvarez/playground/ARMPA/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [10]:
# Import ARMPA modules
try:
    from memory.manager import MemoryManager
    from memory.rl_filter_agent import MemoryFilterPolicy, RLMemoryFilter
    print("✓ Successfully imported ARMPA memory modules")
except ImportError as e:
    print(f"⚠ Warning: Could not import ARMPA modules: {e}")
    print("  Make sure you're running from the ARMPA root directory")

# Try importing WebArena modules
try:
    from browser_env import *
    from agent import *
    print("✓ Successfully imported WebArena modules")
except ImportError as e:
    print(f"⚠ Warning: Could not import WebArena modules: {e}")
    print("  Some cells may not work without WebArena")

/Users/manoloalvarez/playground/ARMPA/.venv/lib/python3.10/site-packages/numpy/_typing/_scalars.py:12: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  _BoolLike_co: TypeAlias = bool | np.bool


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

## 2. Load Sample Trajectory Data

We'll load a sample WebArena trajectory to understand the data structure and how memories are created.

In [ ]:
# Define sample task
SAMPLE_TASK = "What is the top-1 best-selling brand in Quarter 1 2022"

# Check for available trajectory files
trajectory_paths = list(Path('webarena/runs').glob('*/trajectories/*.pkl'))

if trajectory_paths:
    print(f"Found {len(trajectory_paths)} trajectory files:")
    for p in trajectory_paths[:5]:  # Show first 5
        print(f"  {p}")
    
    # Load the first one
    sample_traj_path = trajectory_paths[0]
    print(f"\nLoading: {sample_traj_path}")
    
    with open(sample_traj_path, 'rb') as f:
        trajectory_data = pickle.load(f)
    
    print(f"✓ Loaded trajectory with {len(trajectory_data)} items")
else:
    print("⚠ No trajectory files found in webarena/runs/*/trajectories/")
    print("  You'll need to run data collection first")
    trajectory_data = None

In [ ]:
# Analyze trajectory structure
if trajectory_data:
    print("=== Trajectory Structure Analysis ===")
    print(f"Total items: {len(trajectory_data)}")
    print(f"Items alternate between observations and actions")
    print()
    
    # Sample observation (even indices)
    if len(trajectory_data) > 0:
        sample_obs = trajectory_data[0]
        print("Sample Observation keys:")
        if isinstance(sample_obs, dict):
            for key in sample_obs.keys():
                print(f"  - {key}: {type(sample_obs[key]).__name__}")
    
    # Sample action (odd indices)
    if len(trajectory_data) > 1:
        sample_action = trajectory_data[1]
        print("\nSample Action keys:")
        if isinstance(sample_action, dict):
            for key in sample_action.keys():
                print(f"  - {key}: {type(sample_action[key]).__name__}")
        elif hasattr(sample_action, '__dict__'):
            for key in sample_action.__dict__.keys():
                print(f"  - {key}")

## 3. Memory System - Storage and Retrieval

Let's initialize the memory manager and understand how memories are stored in Qdrant.

In [ ]:
# Initialize Memory Manager
collection_name = "webarena"
mm = MemoryManager(collection_name=collection_name)

print(f"✓ Memory Manager initialized")
print(f"  Collection: {collection_name}")
print(f"  Embedding model: {mm.embedding_model}")
print(f"  Embedding dimension: 384")

In [ ]:
# Check current memory count
try:
    collection_info = mm.client.get_collection(collection_name)
    print(f"Current memories in database: {collection_info.points_count}")
    
    if collection_info.points_count > 0:
        print("\nSample memories:")
        mm.print_all_memories(limit=3)
except Exception as e:
    print(f"Could not retrieve collection info: {e}")

## 4. RL Formulation - State, Action, Reward

### State Space
- Retrieved memory embeddings: `[k, 384]`
- Task embedding: `[384]`
- Current observation embedding: `[384]`
- Agent entropy (uncertainty): `[1]`

### Action Space
- Continuous scores `[0, 1]` for each memory
- Threshold-based selection (e.g., select memories with score > 0.6)

### Reward
- Task success: 1.0 if task completed successfully, 0.0 otherwise
- (Optional) Step efficiency penalty: -0.01 per step

In [ ]:
# Visualize the RL formulation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# State visualization
ax = axes[0]
state_components = ['Memory\nEmbeddings\n[k×384]', 'Task\nEmbedding\n[384]', 'Obs\nEmbedding\n[384]', 'Entropy\n[1]']
state_sizes = [10, 384, 384, 1]
ax.barh(state_components, state_sizes, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax.set_xlabel('Dimension')
ax.set_title('State Components', fontweight='bold')
ax.set_xscale('log')

# Action visualization
ax = axes[1]
k_memories = 10
sample_scores = np.random.beta(2, 2, k_memories)  # Sample scores
threshold = 0.6
colors = ['green' if s > threshold else 'red' for s in sample_scores]
ax.bar(range(k_memories), sample_scores, color=colors, alpha=0.7)
ax.axhline(threshold, color='black', linestyle='--', label=f'Threshold ({threshold})')
ax.set_xlabel('Memory Index')
ax.set_ylabel('Score')
ax.set_title('Action: Memory Scores', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)

# Reward visualization
ax = axes[2]
scenarios = ['Success\n(15 steps)', 'Success\n(30 steps)', 'Failure\n(15 steps)']
rewards = [1.0 - 0.01*15, 1.0 - 0.01*30, 0.0 - 0.01*15]
colors_reward = ['green', 'yellowgreen', 'red']
ax.bar(scenarios, rewards, color=colors_reward, alpha=0.7)
ax.set_ylabel('Total Reward')
ax.set_title('Reward Structure', fontweight='bold')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n=== RL Formulation Summary ===")
print(f"State dimension: 384*3 + 1 = 1153 (plus k memory embeddings)")
print(f"Action dimension: k (one score per memory)")
print(f"Reward: Binary task success + optional step penalty")

## 5. Neural Network Architecture

The `MemoryFilterPolicy` is a neural network that scores memories based on context.

In [ ]:
# Initialize the policy network
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

policy = MemoryFilterPolicy(
    memory_dim=384,
    hidden_dim=256,
    num_heads=4
).to(device)

print(f"\n✓ Policy network initialized")
print(f"  Parameters: {sum(p.numel() for p in policy.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in policy.parameters() if p.requires_grad):,}")

In [ ]:
# Test forward pass with sample data
batch_size = 2
k = 10  # Number of memories

# Create sample inputs
sample_memory_embs = torch.randn(batch_size, k, 384).to(device)
sample_task_emb = torch.randn(batch_size, 384).to(device)
sample_obs_emb = torch.randn(batch_size, 384).to(device)
sample_entropy = torch.rand(batch_size, 1).to(device) * 2  # Entropy in range [0, 2]

print("=== Forward Pass Test ===")
print(f"Input shapes:")
print(f"  Memory embeddings: {sample_memory_embs.shape}")
print(f"  Task embedding: {sample_task_emb.shape}")
print(f"  Observation embedding: {sample_obs_emb.shape}")
print(f"  Entropy: {sample_entropy.shape}")

# Forward pass
with torch.no_grad():
    scores = policy(
        memory_embeddings=sample_memory_embs,
        task_embedding=sample_task_emb,
        obs_embedding=sample_obs_emb,
        entropy=sample_entropy
    )

print(f"\nOutput scores shape: {scores.shape}")
print(f"Score range: [{scores.min():.3f}, {scores.max():.3f}]")
print(f"\nSample scores for batch 0:")
print(scores[0].cpu().numpy())

## 6. Entropy-Based Adaptive Memory Selection

The agent's entropy (uncertainty) determines how many memories to retrieve.
- **High entropy** (uncertain) → Retrieve more memories for guidance
- **Low entropy** (confident) → Retrieve fewer memories to save context

In [ ]:
# Simulate entropy-based memory count selection
def compute_adaptive_k(entropy: float, min_k: int = 3, max_k: int = 15) -> int:
    """Compute number of memories based on entropy"""
    # Normalize entropy to [0, 1] (assuming max entropy ~2.0)
    normalized_entropy = min(entropy / 2.0, 1.0)
    k = min_k + int(normalized_entropy * (max_k - min_k))
    return k

# Test with different entropy values
entropies = np.linspace(0, 2.0, 100)
k_values = [compute_adaptive_k(e) for e in entropies]

plt.figure(figsize=(10, 6))
plt.plot(entropies, k_values, linewidth=2, color='#4ECDC4')
plt.fill_between(entropies, k_values, alpha=0.3, color='#4ECDC4')
plt.xlabel('Agent Entropy (Uncertainty)', fontsize=12)
plt.ylabel('Number of Memories Retrieved (k)', fontsize=12)
plt.title('Adaptive Memory Retrieval Based on Agent Entropy', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add annotations
plt.annotate('Low uncertainty\n→ Few memories', 
             xy=(0.3, 5), fontsize=10, 
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
plt.annotate('High uncertainty\n→ Many memories', 
             xy=(1.6, 13), fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.show()

print("\n=== Adaptive Memory Selection Examples ===")
for e in [0.2, 0.5, 1.0, 1.5, 2.0]:
    k = compute_adaptive_k(e)
    print(f"Entropy {e:.1f} → {k} memories")

## 7. Training Data Format

During data collection (`--collect_rl_data`), we save episode data with:
- Memory embeddings at each step
- Task and observation embeddings
- Agent entropy
- Final task success

In [ ]:
# Look for collected RL training data
rl_data_paths = list(Path('webarena/runs').glob('*/rl_training_data/*.pkl'))

if rl_data_paths:
    print(f"Found {len(rl_data_paths)} RL training data files")
    
    # Load a sample
    sample_data_path = rl_data_paths[0]
    print(f"\nLoading: {sample_data_path}")
    
    with open(sample_data_path, 'rb') as f:
        episode_data = pickle.load(f)
    
    print("\n=== Episode Data Structure ===")
    for key, value in episode_data.items():
        if isinstance(value, list):
            print(f"{key}: list of {len(value)} items")
            if len(value) > 0:
                first_item = value[0]
                if isinstance(first_item, dict):
                    print(f"  First item keys: {list(first_item.keys())}")
                elif hasattr(first_item, 'shape'):
                    print(f"  First item shape: {first_item.shape}")
        else:
            print(f"{key}: {type(value).__name__} = {value}")
else:
    print("⚠ No RL training data found")
    print("  Run data collection with --collect_rl_data flag")
    print("\nExample command:")
    print("  python run.py --test_start_idx 7 --test_end_idx 20 \\")
    print("    --get_memory --store_memory --collect_rl_data \\")
    print("    --num_memories 10 --max_steps 15")

## 8. Training Setup - Behavioral Cloning + PPO

Training strategy:
1. **Behavioral Cloning (BC)**: Pre-train on successful trajectories
2. **PPO Fine-tuning**: Improve with reinforcement learning

In [ ]:
# Example BC pre-training code structure
print("=== Behavioral Cloning Pre-training ===")
print("\nStep 1: Load successful trajectories")
print("  - Filter episodes where success=True")
print("  - Extract (state, action) pairs")
print("\nStep 2: Supervised learning")
print("  - Loss: MSE between predicted scores and 'expert' scores")
print("  - Expert scores: 1.0 for all memories (initially)")
print("  - Or: Use heuristic scores based on memory relevance")
print("\nStep 3: Validation")
print("  - Hold out 20% of successful episodes")
print("  - Monitor loss on validation set")
print("\nSee bc_pretrain.py for full implementation")

In [ ]:
# Example PPO training structure
print("=== PPO Fine-tuning ===")
print("\nHyperparameters:")
hyperparams = {
    'learning_rate': 3e-4,
    'n_steps': 2048,
    'batch_size': 64,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'total_timesteps': 100_000
}

for key, value in hyperparams.items():
    print(f"  {key}: {value}")

print("\nTraining process:")
print("  1. Initialize PPO with BC pre-trained policy")
print("  2. Collect rollouts in WebArena environment")
print("  3. Update policy with PPO objective")
print("  4. Log metrics to TensorBoard")
print("  5. Save checkpoints every 10k steps")
print("\nSee train_rl_filter.py for full implementation")

## 9. Evaluation Metrics

We'll compare three approaches:
1. **No Memory**: Baseline without memory
2. **All Memories**: Include all retrieved memories
3. **RL Filter**: Our learned filter

In [ ]:
# Simulated evaluation results (replace with real results)
results = {
    'Approach': ['No Memory', 'All Memories', 'RL Filter'],
    'Success Rate': [0.45, 0.62, 0.71],
    'Avg Steps': [25.3, 22.1, 18.4],
    'Avg Memories Used': [0, 10, 6.2]
}

df_results = pd.DataFrame(results)

# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Success rate
ax = axes[0]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(df_results['Approach'], df_results['Success Rate'], color=colors, alpha=0.7)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('Task Success Rate', fontweight='bold')
ax.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1%}', ha='center', va='bottom')

# Average steps
ax = axes[1]
bars = ax.bar(df_results['Approach'], df_results['Avg Steps'], color=colors, alpha=0.7)
ax.set_ylabel('Average Steps', fontsize=12)
ax.set_title('Task Efficiency', fontweight='bold')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom')

# Memory usage
ax = axes[2]
bars = ax.bar(df_results['Approach'], df_results['Avg Memories Used'], color=colors, alpha=0.7)
ax.set_ylabel('Avg Memories Used', fontsize=12)
ax.set_title('Memory Efficiency', fontweight='bold')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n=== Evaluation Summary ===")
print(df_results.to_string(index=False))
print("\n✨ RL Filter achieves best success rate with fewer memories!")

## 10. Next Steps

### Immediate Actions
1. **Fix EC2 Shopping Admin** - Resolve redirect issue (see `EC2_ISSUE.md`)
2. **Collect Training Data** - Run 50-100 tasks with `--collect_rl_data`
3. **Train BC Model** - Pre-train on successful trajectories
4. **Fine-tune with PPO** - Improve with reinforcement learning
5. **Evaluate** - Run ablation studies

### Code Files
- `memory/rl_filter_agent.py` - Neural network implementation
- `bc_pretrain.py` - Behavioral cloning pre-training
- `train_rl_filter.py` - PPO training script
- `validate_rl_filter.py` - Evaluation script
- `collect_rl_training_data.py` - Data collection wrapper

### Documentation
- `RL_FILTER_README.md` - Usage guide
- `RL_IMPLEMENTATION_PLAN.md` - Design decisions
- `EC2_ISSUE.md` - Infrastructure status

## Utility: Quick Testing Functions

Helper functions for quick experimentation

In [ ]:
def create_random_episode(k=10, num_steps=5):
    """Create a random episode for testing"""
    episode = {
        'steps': [],
        'success': np.random.random() > 0.5,
        'task': 'Sample task description'
    }
    
    for step in range(num_steps):
        step_data = {
            'memory_embeddings': np.random.randn(k, 384).astype(np.float32),
            'task_embedding': np.random.randn(384).astype(np.float32),
            'obs_embedding': np.random.randn(384).astype(np.float32),
            'entropy': float(np.random.rand() * 2)
        }
        episode['steps'].append(step_data)
    
    return episode

# Test it
test_episode = create_random_episode()
print(f"Created test episode:")
print(f"  Success: {test_episode['success']}")
print(f"  Steps: {len(test_episode['steps'])}")
print(f"  First step memory shape: {test_episode['steps'][0]['memory_embeddings'].shape}")